In [4]:
from embedder import Embedder
import numpy as np
import onnxruntime as ort
from tokenizers import Tokenizer
from pathlib import Path

In [5]:
embedder = Embedder()

In [6]:
query = "How does approximate nearest neighbor search work?"

v = embedder.encode(query)

In [7]:
len(v)

384

In [8]:
print(v[0])

-0.02058203437252893


In [9]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
doc = next(
    d for d in documents
    if d["filename"] == "02-vector-search/lessons/07-sqlitesearch-vector.md"
)

print(doc["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [11]:
page_vector = embedder.encode(doc["content"])

In [13]:
import numpy as np

similarity = np.dot(v, page_vector)
print(similarity)

0.36107026789538205


In [15]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

print(len(chunks))

295


In [16]:
contents = [chunk["content"] for chunk in chunks]

X = embedder.encode_batch(contents)

In [18]:
import numpy as np

X = np.array(X)

scores = X.dot(v)

In [19]:
best_idx = np.argmax(scores)

print(best_idx)
print(scores[best_idx])

94
0.6489016436447387


In [20]:
best_chunk = chunks[best_idx]

print(best_chunk["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [21]:
from minsearch import VectorSearch

In [23]:
import inspect
from minsearch import VectorSearch

print(inspect.signature(VectorSearch))

(keyword_fields=None, numeric_fields=None, date_fields=None)


In [25]:
import minsearch
print(dir(minsearch))

['AppendableIndex', 'DEFAULT_ENGLISH_STOP_WORDS', 'Highlighter', 'Index', 'STEMMERS', 'Tokenizer', 'VectorSearch', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'append', 'filters', 'get_stemmer', 'highlighter', 'lancaster_stemmer', 'minsearch', 'porter_stemmer', 'snowball_stemmer', 'stemmers', 'tokenizer', 'vector']


In [26]:
print(inspect.signature(VectorSearch.fit))
print(inspect.signature(VectorSearch.search))

(self, vectors, payload)
(self, query_vector, filter_dict=None, num_results=10, output_ids=False)


In [28]:
from minsearch import VectorSearch
import numpy as np

# Build the vectors for every chunk
contents = [chunk["content"] for chunk in chunks]
X = np.array(embedder.encode_batch(contents))

# Create the vector index
index = VectorSearch(keyword_fields=["filename"])

# Index vectors together with the chunk metadata
index.fit(X, chunks)


In [29]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embedder.encode(query)

results = index.search(
    query_vector=query_vector,
    num_results=5
)

In [31]:
print(results[0]["filename"])

04-evaluation/lessons/05-search-metrics.md


In [32]:
from minsearch import Index

text_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

text_index.fit(chunks)

In [33]:
text_results = text_index.search(
    query="How do I store vectors in PostgreSQL?",
    num_results=5
)

for r in text_results:
    print(r["filename"])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [34]:
query = "How do I store vectors in PostgreSQL?"
query_vector = embedder.encode(query)

vector_results = index.search(
    query_vector=query_vector,
    num_results=5
)

for r in vector_results:
    print(r["filename"])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [35]:
text_files = {r["filename"] for r in text_results}
vector_files = {r["filename"] for r in vector_results}

print("Only in vector search:")
print(vector_files - text_files)

Only in vector search:
{'02-vector-search/lessons/08-pgvector.md'}


In [36]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [38]:
query = "How do I give the model access to tools?"

In [39]:
query_vector = embedder.encode(query)

vector_results = index.search(
    query_vector=query_vector,
    num_results=5
)

In [40]:
text_results = text_index.search(
    query=query,
    num_results=5
)

In [41]:
results = rrf([vector_results, text_results])

In [42]:
print(results[0]["filename"])

01-agentic-rag/lessons/13-function-calling.md
